# 01 — Gradient boosting fundamentals

**Estimated time:** 100–130 minutes  
**Prerequisites:** notebook 00; decision trees and basic classification metrics.  
**Depends on:** the pre-contact feature contract and development/validation split.

## Learning objectives

- Explain boosting as a sequential additive model rather than a collection of independent trees.
- Show why each new tree follows the negative gradient of a loss function.
- Connect residual fitting in regression to probability errors in binary classification.
- Interpret `n_estimators`, `learning_rate`, and `max_depth` as a capacity tradeoff.
- Evaluate gradient boosting against simple baselines without touching the sealed test set.

## The central idea

A decision tree is a weak learner when deliberately kept shallow. Gradient boosting combines many
such trees **sequentially**. Tree 1 makes a rough prediction; tree 2 learns where the current
ensemble is wrong; tree 3 corrects the remaining error; and so on. Unlike a random forest, these
trees are dependent—later trees cannot be trained before earlier trees exist.

The word *gradient* comes from optimization. At each round, the algorithm calculates the direction
that most rapidly reduces the chosen loss and fits a small tree to approximate that direction.
The learning rate shrinks every correction, trading faster learning for stronger regularization.

## The training process, step by step

1. **Initialize the ensemble.** For binary classification, every row begins with the same constant
   score, usually derived from the positive-class rate. The logistic function converts that score
   into an initial probability.
2. **Measure the current errors.** Compare each observed label with its current probability. Under
   binary log loss, `y - p` gives the direction of the required correction: positive values push
   the score upward and negative values push it downward.
3. **Fit a shallow correction tree.** The new tree partitions rows with similar error signals. Its
   leaves estimate how the current ensemble should change within each region.
4. **Shrink the correction.** Multiply the tree output by `learning_rate`. Small updates make
   learning slower but reduce the chance that one tree overreacts to noise.
5. **Update the ensemble.** Add the shrunken correction to the existing score. The new probability
   comes from the sum of the initial score and every tree contribution learned so far.
6. **Repeat.** Recalculate errors using the updated ensemble, fit the next tree, and continue until
   the tree budget is exhausted or holdout loss stops improving.

During prediction, the model does not calculate training errors again. A row passes through every
learned tree; their shrunken outputs are summed with the initial score, converted to a probability,
and compared with the decision threshold.


### Gradient boosting workflow at a glance

![Gradient boosting workflow: begin with a constant prediction, calculate probability errors, fit a shallow correction tree, shrink its output, update the ensemble, and repeat sequentially.](../assets/gradient_boosting_process.png)

Read the upper process from left to right, then follow the loop back from step 6: after every
update, the next tree sees newly calculated errors. The lower comparison highlights the structural
difference from a random forest. Random-forest trees are independent and can train in parallel;
boosting trees form an ordered chain because each tree depends on the current ensemble. The values
shown are pedagogical examples rather than outputs from the Bank Marketing experiment.


In [ ]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# Known interoperability/UI warnings do not affect predictions or notebook execution.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, confusion_matrix, log_loss,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # Return the course root when present, otherwise the notebook's directory.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # Set FAST_MODE=0 for full-size experiments; laptop mode is the default.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # The course ships with a local dataset; notebooks never access the network.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and drop leakage columns by default.'''
    # Load the data, encode y, and exclude post-call duration by default.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    if not include_duration:
        frame = frame.drop(columns=LEAKAGE_COLUMNS)
    return frame

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # Deterministic stratified 60/20/20 split; test stays sealed until notebook 09.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # Preprocessing is fitted only inside the enclosing training/CV pipeline.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute ranking and threshold-based classification metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"roc_auc": roc_auc_score(y_true, probability),
            "pr_auc": average_precision_score(y_true, probability),
            "log_loss": log_loss(y_true, probability),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "cost": float(fp + 5 * fn)}

def threshold_table(y_true, probability, thresholds=None):
    '''Evaluate classification metrics across a list of decision thresholds.'''
    thresholds = np.linspace(0.05, 0.80, 76) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t),
                          **classification_metrics(y_true, probability, float(t))}
                         for t in thresholds])

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             log_loss, ConfusionMatrixDisplay)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

development, validation, _sealed_test = make_splits(load_bank_data(), reduced=FAST_MODE)
X_dev, y_dev = split_xy(development)
X_val, y_val = split_xy(validation)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
CV_SPLITS = list(cv.split(X_dev, y_dev))
print({"development": X_dev.shape, "validation": X_val.shape,
       "positive_rate": round(y_dev.mean(), 3)})


## Build boosting by hand: squared-error regression

Regression gives the cleanest intuition. Under squared-error loss, the negative gradient is the
familiar residual `actual - prediction`. We start with the mean target, fit a shallow tree to the
residuals, shrink its output, add it to the current prediction, and repeat.

\[
F_m(x) = F_{m-1}(x) + \eta h_m(x)
\]

Here, `h_m` is the new tree and `η` is the learning rate. The ensemble—not the newest tree—is the
model.


In [ ]:
rng = np.random.default_rng(SEED)
x = np.linspace(-3, 3, 160)
y = np.sin(x) + 0.25 * x + rng.normal(0, 0.12, len(x))
X_toy = x.reshape(-1, 1)

learning_rate = 0.25
prediction = np.full_like(y, y.mean())
snapshots = {0: prediction.copy()}
toy_trees = []

for round_number in range(1, 11):
    residual = y - prediction
    tree = DecisionTreeRegressor(max_depth=1, random_state=SEED + round_number)
    tree.fit(X_toy, residual)
    prediction += learning_rate * tree.predict(X_toy)
    toy_trees.append(tree)
    if round_number in {1, 3, 10}:
        snapshots[round_number] = prediction.copy()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for axis, (round_number, stage_prediction) in zip(axes, snapshots.items()):
    axis.scatter(x, y, s=9, alpha=0.35, label="observations")
    axis.plot(x, stage_prediction, color="tab:orange", linewidth=2, label="ensemble")
    axis.set_title(f"after {round_number} trees")
    axis.set_xlabel("x")
axes[0].set_ylabel("target")
axes[0].legend(loc="upper left")
fig.suptitle("Each stump adds a small correction to the ensemble", y=1.04)
plt.tight_layout()
plt.show()


We can also look at the *errors* directly. Boosting does not replace the whole model at each
step; it keeps the current ensemble and asks the next stump to explain what is still left in the
residuals.


In [ ]:
# Compare the first correction step with the final residuals.
residual_0 = y - snapshots[0]
first_tree = toy_trees[0]
first_correction = learning_rate * first_tree.predict(X_toy)
residual_1 = y - snapshots[1]
residual_final = y - snapshots[10]

fig, axes = plt.subplots(2, 2, figsize=(14, 6.2), sharex=True)
axes = axes.ravel()

axes[0].scatter(x, residual_0, s=9, alpha=0.35, color="tab:blue")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("start: residuals from mean")

axes[1].scatter(x, residual_0, s=9, alpha=0.18, color="tab:blue", label="residual before 1 tree")
axes[1].scatter(x, residual_1, s=9, alpha=0.35, color="tab:green", label="residual after 1 tree")
axes[1].plot(x, first_correction, color="tab:orange", linewidth=2, label="first tree correction")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("one stump reduces the error")

step_idx = np.linspace(10, len(x) - 10, 10, dtype=int)
axes[2].plot(x, snapshots[0], color="tab:gray", linewidth=1.4, label="initial prediction")
axes[2].plot(x, snapshots[1], color="tab:orange", linewidth=2, label="after 1 tree")
axes[2].plot(x, snapshots[10], color="tab:red", linewidth=2, label="after 10 trees")
for idx in step_idx:
    axes[2].annotate(
        "",
        xy=(x[idx], snapshots[1][idx]),
        xytext=(x[idx], snapshots[0][idx]),
        arrowprops=dict(arrowstyle="->", color="tab:orange", lw=1),
    )
axes[2].axhline(y.mean(), color="black", linewidth=1, linestyle=":")
axes[2].set_title("arrows show the correction")

axes[3].scatter(x, residual_final, s=9, alpha=0.35, color="tab:blue", label="residual after 10 trees")
axes[3].axhline(0, color="black", linewidth=1)
axes[3].set_title("after 10 trees")

axes[0].set_ylabel("residual / correction")
axes[2].set_ylabel("prediction")
axes[3].set_ylabel("residual / correction")
for axis in axes:
    axis.set_xlabel("x")
axes[1].legend(loc="upper left")
axes[2].legend(loc="upper left")
fig.suptitle("Gradient boosting keeps shrinking the remaining errors", y=1.02)
plt.tight_layout()
plt.show()


The staircase shape is expected: shallow trees produce piecewise-constant corrections. More
rounds create a flexible function. That flexibility is useful until the ensemble starts fitting
noise.

## From residuals to classification gradients

Binary classification usually minimizes log loss. The model maintains a score that becomes a
probability through the logistic function. For a training row, the negative gradient is closely
related to `observed label - current probability`:

- a positive row predicted at `0.10` needs a large upward correction (`1 - 0.10 = 0.90`);
- a negative row predicted at `0.90` needs a large downward correction (`0 - 0.90 = -0.90`);
- a correct, confident row needs only a small correction.

This is why describing boosting as “fitting mistakes” is useful, but incomplete: the precise
mistake depends on the differentiable loss function.


In [ ]:
gradient_example = pd.DataFrame({
    "observed_y": [1, 0, 1, 0],
    "current_probability": [0.10, 0.90, 0.80, 0.15],
})
gradient_example["negative_gradient_y_minus_p"] = (
    gradient_example["observed_y"] - gradient_example["current_probability"]
)
gradient_example["interpretation"] = [
    "strong upward correction", "strong downward correction",
    "small upward correction", "small downward correction",
]
gradient_example


## Three controls govern model capacity

| Parameter | Meaning | If too small | If too large |
|---|---|---|---|
| `n_estimators` | number of sequential trees | underfitting | overfitting and extra compute |
| `learning_rate` | size of each tree's contribution | needs more trees | corrections can be unstable |
| `max_depth` | interactions each tree can represent | misses structure | fits noise and becomes harder to explain |

`learning_rate` and `n_estimators` must be considered together. A smaller learning rate typically
needs more trees. Shallow trees are not a weakness here; they are intentional components whose
combined output can represent complex nonlinear structure.


In [ ]:
def make_dense_preprocessor(frame):
    categorical = frame.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = frame.select_dtypes(include=np.number).columns.tolist()
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numerical),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])

def make_model(model):
    return Pipeline([
        ("preprocess", make_dense_preprocessor(X_dev)),
        ("model", model),
    ])

candidates = {
    "always no": make_model(DummyClassifier(strategy="most_frequent")),
    "logistic regression": make_model(LogisticRegression(max_iter=1200, random_state=SEED)),
    "gradient boosting": make_model(GradientBoostingClassifier(
        n_estimators=100 if FAST_MODE else 180,
        learning_rate=0.05, max_depth=2, random_state=SEED,
    )),
}

rows = []
for name, estimator in candidates.items():
    scores = cross_validate(
        estimator, X_dev, y_dev, cv=CV_SPLITS,
        scoring=["accuracy", "balanced_accuracy", "f1", "neg_log_loss"], n_jobs=-1,
    )
    rows.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "balanced_accuracy": scores["test_balanced_accuracy"].mean(),
        "f1": scores["test_f1"].mean(),
        "log_loss": -scores["test_neg_log_loss"].mean(),
        "accuracy_sd": scores["test_accuracy"].std(ddof=1),
    })
cv_results = pd.DataFrame(rows).set_index("model")
cv_results


Accuracy remains easy to communicate, but it must be read beside balanced accuracy and F1: the
majority baseline is accurate on this imbalanced dataset while finding no subscribers. Log loss
evaluates probability quality and is also the objective optimized by the classifier.

## Watch training and holdout loss across boosting rounds

Early trees reduce both training and holdout loss. Eventually, training loss may continue falling
while holdout loss stops improving. The best iteration is selected on an internal subset of the
development data—not the course validation set.


In [ ]:
X_fit, X_stop, y_fit, y_stop = train_test_split(
    X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=SEED
)
curve_preprocessor = make_dense_preprocessor(X_fit)
X_fit_t = curve_preprocessor.fit_transform(X_fit)
X_stop_t = curve_preprocessor.transform(X_stop)

curve_model = GradientBoostingClassifier(
    n_estimators=220 if FAST_MODE else 400,
    learning_rate=0.05, max_depth=2, random_state=SEED,
).fit(X_fit_t, y_fit)

train_loss = [log_loss(y_fit, p[:, 1]) for p in curve_model.staged_predict_proba(X_fit_t)]
stop_loss = [log_loss(y_stop, p[:, 1]) for p in curve_model.staged_predict_proba(X_stop_t)]
best_iteration = int(np.argmin(stop_loss) + 1)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(train_loss) + 1), train_loss, label="internal training")
plt.plot(range(1, len(stop_loss) + 1), stop_loss, label="internal holdout")
plt.axvline(best_iteration, color="black", linestyle="--",
            label=f"best iteration = {best_iteration}")
plt.xlabel("number of trees")
plt.ylabel("log loss")
plt.title("Boosting learning curve")
plt.legend()
plt.show()


## Refit once, then evaluate validation once

After selecting the tree count internally, we rebuild the full pipeline on all development rows.
Validation is used only for the final lesson-level check. The sealed test set remains untouched.


In [ ]:
final_boosting = make_model(GradientBoostingClassifier(
    n_estimators=best_iteration, learning_rate=0.05,
    max_depth=2, random_state=SEED,
)).fit(X_dev, y_dev)
validation_probability = final_boosting.predict_proba(X_val)[:, 1]
validation_prediction = validation_probability >= 0.5

validation_metrics = pd.Series({
    "accuracy": accuracy_score(y_val, validation_prediction),
    "balanced_accuracy": balanced_accuracy_score(y_val, validation_prediction),
    "f1": f1_score(y_val, validation_prediction, zero_division=0),
    "log_loss": log_loss(y_val, validation_probability),
}, name="gradient boosting validation")
display(validation_metrics.to_frame())
ConfusionMatrixDisplay.from_predictions(
    y_val, validation_prediction, display_labels=["no", "yes"], colorbar=False
)
plt.title("Validation confusion matrix at threshold 0.5")
plt.show()


## Gradient boosting, random forests, and CatBoost

- A **random forest** trains many independent trees on randomized samples/features and averages
  them. Parallelism and variance reduction are central.
- **Gradient boosting** trains dependent trees sequentially, with each tree reducing the current
  ensemble's loss. Bias reduction and careful regularization are central.
- **CatBoost** is a gradient-boosting implementation with specialized categorical statistics and
  techniques that reduce prediction shift. Notebook 02 builds on the boosting logic established
  here before explaining CatBoost's categorical machinery.

## Common mistakes and leakage warnings

- Calling each tree a complete model instead of a correction added to the ensemble.
- Increasing tree count without considering learning rate and depth.
- Selecting the best iteration on the final validation or test set and reporting the same score.
- Fitting one-hot encoding on all rows before cross-validation.
- Using accuracy alone on an imbalanced target.
- Assuming lower training loss guarantees better generalization.

## Exercises

1. Change the toy learner from `max_depth=1` to `max_depth=2`. Describe how the staircase changes.
2. Compare `(learning_rate=0.1, n_estimators=50)` with `(0.025, 200)` using identical CV folds.
3. Plot validation balanced accuracy, not only log loss, over boosting stages.
4. **Challenge:** implement one boosting round for binary log loss by fitting a regression stump
   to `y - p`, then update the raw scores and probabilities.

## Summary

Gradient boosting is gradient descent in function space: each shallow tree approximates a
loss-reducing correction, and the ensemble adds those corrections gradually. Tree count, learning
rate, and depth jointly control capacity. Internal early stopping chooses complexity; cross-
validation estimates generalization; validation remains a final check. This foundation makes
CatBoost's ordered categorical statistics in notebook 02 easier to understand.

## References

- [scikit-learn gradient boosting user guide](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosting)
- [GradientBoostingClassifier API](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)
- [Friedman (2001), Greedy Function Approximation: A Gradient Boosting Machine](https://doi.org/10.1214/aos/1013203451)
- [scikit-learn model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
